# PyTorch Pipeline

In this notebook, we walk through a complete PyTorch pipeline to train and evaluate a simple fully connected neural network for an image classification task using the Devanagari Character Set. 
This notebook is designed to help you understand the key components of `PyTorch` package, including:

- PyTorch Dataset and DataLoader structure
- Network architecture definition
- Loss function, optimizer and learning rate scheduler setup
- Network training
- Inference

**What is PyTorch?**

**PyTorch** is an open-source deep learning framework developed by Meta AI. 
It provides a flexible and intuitive interface for building, training, and deploying machine learning models. 
PyTorch is especially popular in research and education due to its dynamic computation graph, which makes debugging and experimentation easier. 
You can explore the full PyTorch documentation here: https://pytorch.org/docs/stable/index.html

**Dataset Overview**

The Devanagari Character Set is a collection of 92000 handwritten character images from the Devanagari script, which is used in several South Asian languages including Hindi, Nepali, and Sanskrit. 
The dataset includes *46* classes representing various consonants and digits, and contains grayscale images of size *32x32* pixels. 
Our goal is to build a model that can classify these characters accurately with `PyTorch`.

**Workflow Summary**

This notebook will guide you through the following steps:

1. Data Loading
    - Download or access the Devanagari dataset
    - Read image data and labels

2. Data Analysis & Visualization
    - Explore class distribution
    - Visualize sample characters

3. Train-Validation-Test Split
    - Organize data into training, validation and test sets

4. Dataset and DataLoader Implementation
    - Create a custom `torch.utils.data.Dataset` class
    - Apply data transformations (e.g., normalization)
    - Use `torch.utils.data.DataLoader` for batching and shuffling

5. Model Definition
    - Build a simple fully connected neural network by inheriting from `torch.nn.Module`

6. Loss Function,  Optimizer and Learning Rate Scheduler Setup
    - Choose an appropriate loss function from `torch.nn`
    - Use `torch.optim` to configure the optimization algorithm for training
    - Set up a learning rate scheduler using `torch.optim.lr_scheduler` to adjust the learning rate over time during training

7. Model Training
    - Setting `train` & `eval` modes of the model
    - Loop through epochs
    - Track loss and accuracy for the training and validation sets

8. Evaluation on Test Set & Inference
    - Run predictions on the test set
    - Evaluate accuracy and visualize model predictions

## Setup and Imports
We use some non-standard packages here, so first install them via pip.

In [ ]:
!pip install kagglehub torchinfo

Let’s start by importing the necessary libraries and setting up our environment.

In [ ]:
# Core libraries
import torch

# some extra libraries used in this example
import pandas
import numpy
import torchinfo

import kagglehub
import zipfile

# for sklearn, we need to import sub-packages to access their functionality
import sklearn.model_selection
import sklearn.metrics

# Plotting settings
from matplotlib import pyplot
pyplot.rcParams['image.cmap'] = 'gray'


## Step 1: Data Loading

In this step, we download the Devanagari Character Dataset from Kaggle using the `kagglehub` package. 
This tool allows direct access to certain public Kaggle datasets without requiring an account or API key. 
The dataset is automatically downloaded, and cached locally, so repeated runs won't trigger another download.

In [ ]:
# Download only the 'data.csv' file from the public dataset
data_path = kagglehub.dataset_download(
    handle="rishianand/devanagari-character-set/versions/4",
    path="data.csv"
)
print("Data downloaded to:",data_path)

# Unzip it
with zipfile.ZipFile(data_path, 'r') as zip_ref:
    zip_ref.extractall()  # or any output directory you like

## Step 2: Data Analysis & Visualization

Instead of reading raw image files, we use a `.csv` file provided. 
It contains the pixel values for each image along with their corresponding labels. 
Each row in the CSV represents one image, where the pixel values are flattened into a single row, and the label (the last column) indicates the character class.

In this step, we begin by extracting the relevant parts from the CSV file to separate the `inputs` and `targets`, which we'll later use to build our custom dataset and dataloader for training. 
Before jumping into model building, it is essential to explore and understand the data. 
This helps us get familiar with the structure, scale, and variety of the dataset.

First, we use `pandas` to read the data and get an understanding of its content.

In [ ]:
# Read the data
data = pandas.read_csv("data.csv")

# Display data structure
data.head()

Well, that doesn't help us much, but the headers seem to mention that we actually include images with pixels here. 
The last column `character` in the dataset seems to be the label.
Let's extract this data and show the distributions of the classes.

In [ ]:
# Use all but the last column as input
inputs = data.iloc[:,:-1].to_numpy()
# Use the "character" column as our labels
targets = data["character"].to_numpy()

# Count the occurrences of each label
unique_labels, counts = numpy.unique(targets, return_counts=True)

# Display dataset information
print(f"Dataset shape: {inputs.shape}")
print(f"The range of pixel values: ({inputs.min()},{inputs.max()})")
print(f"Number of classes: {len(unique_labels)}")

# Check distribution of characters
pyplot.bar(unique_labels, counts, width=0.8,color='#87CEFA')
pyplot.xlabel('Class Label')
pyplot.ylabel('Number of Samples')
pyplot.title('Distribution of Devanagari Character Classes')
pyplot.xticks(unique_labels,rotation=90)  # Show all labels on x-axis
pyplot.grid(axis='y', linestyle='--', alpha=0.6)
pyplot.tight_layout()
pyplot.show()

Alright, the data seems to be well-balanced, all samples appear in the same frequency.
Let's now take a look into some of the examples. 
Since they seem to be images, let us convert them to 2D and display one sample for each class.
Note that this conversion is only for illustration, the original samples will still be vectorial.

In [ ]:
# Visualize an image for each charecter
fig, axes = pyplot.subplots(4, 12, figsize=(16, 6))
axes = axes.flatten()

# For each class label, pick the first image and show it
for idx,tau in enumerate(unique_labels):

    index = numpy.where(targets == tau)[0][0]
    # Reshape back to 32x32
    image = inputs[index].reshape(32, 32)

    axes[idx].imshow(image)
    axes[idx].set_title(f"{tau}", fontsize=8)
    axes[idx].axis('off')

# Hide any remaining unused subplots (since 4x12 = 48 > 46)
for i in range(len(unique_labels), len(axes)):
    axes[i].axis('off')

pyplot.suptitle("One Sample per Devanagari Character", fontsize=16)
pyplot.tight_layout()
pyplot.show()

## Step 3: Train-Validation-Test Split

In this step, we split our data into three parts: training ($70\%$), validation ($15\%$) and test ($15\%$) by using `sklearn`.
Since the original distribution of labels is balanced, we use a stratified sampling to keep this property.

In [ ]:
# First split: 70% train, 30% val-test
X_train, X_val_test, T_train, T_val_test = sklearn.model_selection.train_test_split(inputs, targets, test_size=0.3, random_state=42, stratify=targets)

# Second split: 50% validation, 50% test (resulting in 70%/15%/15% overall)
X_val, X_test, T_val, T_test = sklearn.model_selection.train_test_split(X_val_test, T_val_test, test_size=0.5, random_state=42, stratify=T_val_test)

# Display the shapes of the resulting datasets
print(f"Training set: {X_train.shape}, {T_train.shape}")
print(f"Validation set: {X_val.shape}, {T_val.shape}")
print(f"Test set: {X_test.shape}, {T_test.shape}")

## Step 4: Dataset and DataLoader Implementation

In this step, we prepare the data for training by implementing a custom dataset class based on `torch.utils.data.Dataset`, and wrapping it in a `DataLoader` using `torch.utils.data.DataLoader` to organize the samples into batches. 
PyTorch's `Dataset` class gives us full control over how to load and preprocess the data. 

Since we implement a *fully connected network* (instead of a CNN), the model expects **1D input vectors** rather than 2D image tensors. 
Fortunately, because we load the data from a `.csv` file where each image is stored as a flattened row of pixel values, no additional reshaping is required during data preparation. 
Each input sample should be: $\vec x^{[n]} \in \mathbb R^{1024}$ representing a $32\times32$ grayscale image flattened into a vector, and the corresponding target label is a class index: $\tau^{[n]} \in \{0,\ldots,45\}$.

To prepare the input data for training, we also apply a simple transformation that scales pixel values $x_d$ from the range $[0, 255]$ to $[0, 1]$. 
This normalization helps the model train more efficiently by ensuring that all input values are within a similar, smaller numerical range — which stabilizes gradient updates and speeds up convergence. 
**It is of utmost importance that all datasets (train/val/test) apply the exact same data normalization.**

1. To make the dataset compatible with PyTorch's data loading pipeline, we implement three essential methods in the custom `DevanagariDataset`:
    - `__init__`: Initializes the dataset with input data (`X`) and labels (`T`), and optionally applies any preprocessing (e.g., normalization or data augmentation).
    - `__getitem__`: Returns a single sample (input vector and corresponding label) at the given index. This is the method the `DataLoader` uses to retrieve each batch of data during training.
    - `__len__`: Returns the total number of samples in the dataset. This is used by the `DataLoader` to know how many batches to create.
2. Then, we wrap the `Dataset` with a `DataLoader` to handle the delivery of data to the model during training and evaluation. It automates batching, shuffling, and parallel data loading, making the training loop both cleaner and more efficient. Key parameters in `DataLoader` are:
    - `dataset`: The dataset object that provides samples.
    - `batch_size`: Number of samples per batch (e.g., 64). Controls how many examples the model sees at once during training.
    - `shuffle`: If True, the data is shuffled at the beginning of each epoch (highly recommended for training).
    - `num_workers` (optional): Number of subprocesses to use for data loading. More workers can speed up loading on machines with multiple CPU cores. Too many workers might also slow down the process, especially when the data transform is simple. Since our `transform` is simple, we do not need parallelization.

In [ ]:
# Build custom dataset
class DevanagariDataset(torch.utils.data.Dataset):
    def __init__(self, X, T, transform=None):

        # Convert to tensors if needed
        if not isinstance(X, torch.Tensor):
            X = torch.tensor(X, dtype=torch.float32)

        # Store inputs
        self.inputs = X

        # Encode string labels to integer class indices
        unique_labels = sorted(set(T))
        self.label2index = {label: idx for idx, label in enumerate(unique_labels)}
        # Store encoded targets
        self.targets = torch.tensor([self.label2index[label] for label in T], dtype=torch.long)

        # Store transform
        self.transform = transform


    def __getitem__(self, index):
        # get input and target for the given index
        input = self.inputs[index]
        target = self.targets[index]

        # Apply optional transform (e.g., normalization)
        if self.transform:
            input = self.transform(input)

        # return both the input sample and its target class
        return input, target


    def __len__(self):
        # return the number of samples in our dataset
        return len(self.inputs)


# Define the transform: Normalize pixel values from [0, 255] to [0, 1]
normalize = lambda x: x / 255.0

# Create Dataset objects for all sets
# Make sure to use the exact same transform for each set
train_dataset = DevanagariDataset(X_train ,T_train, transform=normalize)
val_dataset = DevanagariDataset(X_val, T_val, transform=normalize)
test_dataset = DevanagariDataset(X_test, T_test, transform=normalize)

batch_size = 256  # You can change this as needed

# Create DataLoaders for the three datasets.
# Here, we only set shuffle to true for the training loader
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader  = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## Step 5: Model Definition

In this step, we define our neural network by creating a custom class that inherits from `torch.nn.Module`, which is the base class for all neural networks in PyTorch. 
This approach gives us full control over the architecture and forward computation of our network. 
Since we are working with a fully connected network, the input to the model is a flattened vector of 1024 features (from a $32\times32$ grayscale image), and the output is a class index for each Devanagari character. 
To implement any custom model using `torch.nn.Module`, two essential methods must be defined:
- `__init__`: Defines the layers of the model. This is where we declare fully connected layers, activation functions, dropout, etc.
- `forward`: Defines the forward pass of the input through the network — how data flows from input to output. 

In our implementation below, we make use of a mixture of submodules, in our case layers with learnable parameters, and *inline* functions defined in `torch.nn.functional`. 
Please note that inline functions cannot have learnable parameters.

Our fully connected network has the following layers:

1. A fully-connected `torch.nn.Linear` layer with $K^{(1)}=512$ hidden units
2. `ReLU` activation via `torch.nn.functional.relu`. Alternatively, you can also define `torch.nn.ReLU` as variable in your module.
3. Another fully-connected `torch.nn.Linear` layer with $K^{(2)}=256$ hidden units
4. A Batch normalization layer. Here, we have a one-dimensional input, which requires `torch.nn.BatchNorm1d`.
5. `ReLU` activation
6. A `torch.nn.Dropout` layer with the probability of $\kappa=0.4$
7. Linear `torch.nn.Linear` with $O$ outputs

Here, we include two layers (batchnorm and dropout) which behave differently during training and evaluation.
Therefore, we need to ensure that the model is in the correct `mode` by setting it to `train` and `eval` mode in the training loop below.

Please note that the definition of the layers in our network does not follow the exact order of application of the layers.
This is to showcase that you do not need to define them in applied order.
It is, however, recommended defining them in the correct order to increase readability.

In [ ]:
class FullyConnectedNet(torch.nn.Module):
    def __init__(self, input_size, K1, K2, O, dropout_p):
        # Call base class constructor
        super(FullyConnectedNet, self).__init__()

        # Define layers explicitly using torch.nn
        self.fc1 = torch.nn.Linear(input_size, K1)
        self.fc2 = torch.nn.Linear(K1, K2)
        self.fc3 = torch.nn.Linear(K2, O)

        # We want a batchnorm after the second layer
        self.batchnorm = torch.nn.BatchNorm1d(K2)

        # And a dropout
        self.dropout = torch.nn.Dropout(dropout_p, inplace=False)

    def forward(self, x):

        if x.ndim > 2:
            x = torch.flatten(x, start_dim=1)  # Flatten if needed

        # run through first layer and activation function
        # Here, we use the functional form of torch.nn.ReLU,
        # which is possible since torch.nn.ReLU has no learnable parameters
        h1 = torch.nn.functional.relu(self.fc1(x), inplace=True)

        # run through second layer, batchnorm and activation function
        h2 = torch.nn.functional.relu(self.batchnorm(self.fc2(h1)), inplace=True)

        # apply dropout and the final fully-connected layer to produce the logits
        z = self.fc3(self.dropout(h2))

        return z

Let's create an instance of our network and print its layers.

In [ ]:
# Create model
model = FullyConnectedNet(input_size=1024, K1=512, K2=256, O=46, dropout_p=0.4)

print(model)

As you can see, the layers are listed in the *order of definition*, not in the *order of application*. 
The `torchinfo` package provides a convenient way to display a layer-by-layer summary of a PyTorch model.  
Using the `torchinfo.summary` function, we can view the layers **in the order of the forward pass**, along with their output shapes, number of trainable parameters, and the overall model size.

In [ ]:
# Summary of the model including forward pass
torchinfo.summary(model, input_data=torch.empty((1,1024)),device="cpu")

## Step 6: Loss Function,  Optimizer and Learning Rate Scheduler Setup

Before training the model, we need to configure three essential components: the computation **device**, the **loss function**, and the **optimizer**.
Additionally, we select a learning rate scheduler to speed up learning.

- `Device`: To take advantage of GPU acceleration (if available), we define a device variable using `torch.device`.
- `Loss`: Since this is a categorical classification problem with $O=46$ character classes, we use *categorical cross entropy loss* from `torch.nn.CrossEntropyLoss` function. This loss function is ideal for classification tasks where the model outputs *raw logits* (not probabilities), and the targets are *integer-encoded* class labels. It internally applies `Softmax` followed by `Negative Log Likelihood` in a numerically stable and efficient way. One important parameter to configure is `reduction`, which determines how the individual sample losses are aggregated:
    - `"mean"` (default): returns the *average* loss across the batch
    - `"sum"`: returns the *total* (unscaled) loss across the batch
    - `"none"`: returns the loss *per sample* (no reduction applied); Be aware that `"none"` is different from `None`

- `Optimizer`: To update the model's parameters during training, we need an optimizer. PyTorch provides several built-in optimizers under the `torch.optim` module, including popular choices like *SGD*, *Adam*, and *RMSprop*. For this project, we make use of **Stochastic Gradient Descent** (SGD) utilizing `torch.optim.SGD`. Key parameters for SGD are:
    - `params`: the model’s parameters that needs to be updated (usually passed as `model.parameters()`)
    - `lr` (learning rate): controls how big the update steps are — crucial for convergence
    - `momentum` (optional): adds a velocity term to the updates, helping the model move faster in the right direction
    - `weight_decay` (optional): applies L2 regularization to help prevent overfitting

- `Learning Rate Scheduler`: In addition to setting up the optimizer, we often use a learning rate scheduler to adjust the learning rate during training. While a constant learning rate might work for some problems, dynamically updating it can lead to *faster convergence*. Schedulers are especially useful when the model stops improving on the validation set. For this pipeline, we use `torch.optim.lr_scheduler.ReduceLROnPlateau`, which is a great general-purpose scheduler. It reduces learning rate when a monitored metric (like validation loss/accuracy) has stopped improving. Key parameters for `ReduceLROnPlateau` are:
    - `optimizer`: The optimizer whose learning rate we want to schedule. Required so the scheduler can access and update the learning rate of the optimizer.
    - `mode`: Defines whether the scheduler should look for a **minimum** or **maximum** in the monitored metric. Use `min` (default) to reduce the LR when the validation loss stops decreasing. Use `max` if you're monitoring something like accuracy and want to reduce when it stops increasing.
    - `factor`: The multiplicative factor by which the learning rate will be reduced. For example, `factor=0.1` (default) means the learning rate will be multiplied by 0.1.
    - `patience`: Number of epochs to wait after the last improvement before reducing the learning rate. This helps prevent reacting to small fluctuations in the monitored metric.
    - `min_lr`: The lower bound on the learning rate. Once the LR reaches this value, it won’t be reduced further — helps avoid reducing LR to near-zero.

In [ ]:
# Define the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Define the loss function
loss_function = torch.nn.CrossEntropyLoss(reduction="mean")

# Define the optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# Define the LRScheduler
# Here we will monitor the validation accuracy, so mode needs to be set as "max"
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.1, patience=2, min_lr=1e-4)

## Step 7: Model Training

Now that the dataloaders, model, loss function, and optimizer are defined, we move on to training the model over multiple epochs. 
At the end of each epoch, we also evaluate the model on the validation set to monitor its generalization performance.

The training loop in PyTorch typically includes the following steps:
1. **Send the model to the selected device**:
Before training begins, we move the model to the computation device (CPU or GPU) using the defined `device`.

2. **Set the model to training mode** (optional): 
This enables training-specific behavior for layers like `Dropout` and `BatchNorm` using `model.train()`.

3. **Iterate over the training data using its** `DataLoader`. For each mini-batch:
    - Move inputs and labels to the same device as the model
    - Clear old gradients with `optimizer.zero_grad()`
    - Perform a forward pass to get the logits for the current batch
    - Compute the loss `J` using the loss function
    - Backpropagate with `J.backward()` to get the gradients of all learnable parameters
    - Update weights using `optimizer.step()`
    - Store the training loss/accuracy using `J.item()`

4. **Run a validation pass after each epoch if validation set is available**. After training on all batches, we evaluate the model on the validation set:
    - Set the model to evaluation mode using `model.eval()` if the model contains layers like `Dropout` and `BatchNorm`.
    - Use `with torch.no_grad()` to disable gradient tracking
    - Iterate over the validation data using its `DataLoader` to get predictions
    - Compute validation loss/accuracy to track progress

5. **When using a scheduler, update the learning rate based on validation performance utilizing** `lr_scheduler.step()`:
This checks whether the monitored validation metric (e.g., loss/accuracy) has stopped improving and reduces the learning rate if needed, helping the model converge better by allowing smaller, more stable updates in later epochs.

This training–validation cycle is repeated for multiple epochs, helping us observe both learning and potential overfitting.
On the GPU, this training might run for a few minutes, on the CPU it might take a little longer.

In [ ]:
num_epochs = 60

# Send model to the computation device
model.to(device)

# Store loss values and accuracies for both sets
train_loss_list, train_acc_list = [], []
val_loss_list, val_acc_list = [], []

# Train the model over epochs
for epoch in range(num_epochs):
    # Set the training mode
    model.train()
    train_loss, train_correct = 0,0

    # Get current learning rate from optimizer (only for displaying purposes)
    current_lr = optimizer.param_groups[0]['lr']

    for x, t in train_loader:
        # Clear old gradients
        optimizer.zero_grad()
        # Send the data to device
        x, t = x.to(device), t.to(device)
        # Forward pass
        z = model(x)
        # Calculate the loss
        J = loss_function(z, t)
        # Backpropagation
        J.backward()
        # Weight update
        optimizer.step()
        # Update the epoch loss
        train_loss += J.item() * len(t)
        # Count correctly classified samples within the batch
        preds = torch.argmax(z, dim=1)
        train_correct += (preds == t).sum().item()

    # Store training accuracy and loss for current epoch
    avg_train_loss = train_loss/len(train_dataset)
    train_acc = train_correct/len(train_dataset)
    train_loss_list.append(avg_train_loss)
    train_acc_list.append(train_acc)

    # Validate network on validation data
    model.eval()
    val_loss, val_correct = 0, 0
    with torch.no_grad():
        for x, t in val_loader:
            # Send the data to device
            x,t = x.to(device), t.to(device)
            # Get predictions
            z = model(x)
            # Compute validation loss
            J = loss_function(z,t)
            # Update the validation loss
            val_loss += J.item() * len(t)
            # Update the validation corrects
            preds = torch.argmax(z, dim=1)
            val_correct += (preds == t).sum().item()

    # Store validation accuracy and loss for current epoch
    avg_val_loss = val_loss/len(val_dataset)
    val_acc = val_correct/len(val_dataset)
    val_loss_list.append(avg_val_loss)
    val_acc_list.append(val_acc)

    # LR scheduler step
    lr_scheduler.step(val_acc)

    # --- Logging ---
    print(f"Epoch {epoch+1:>2}/{num_epochs} | "
          f"LR: {current_lr:<6} | "
          f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")

To display the training and generalization performance, we plot the losses and accuracies for the training and test sets after training has finished.

Note that there exist methods to visualize such measures during training, for example, using `tensorboard`.
For our small example here, it is out of scope to explain how this works.
For larger projects, it is surely advisable to read the `tensorboard` documentation: https://www.tensorflow.org/tensorboard and understand how to use `tensorboard` with `pytorch`: https://pytorch.org/tutorials/recipes/recipes/tensorboard_with_pytorch.html

In [ ]:
pyplot.figure(figsize=(12, 6))
# --- Loss plot ---
pyplot.subplot(1, 2, 1)
pyplot.plot(train_loss_list, label="Train Loss", marker='o')
pyplot.plot(val_loss_list, label="Val Loss", marker='o')
pyplot.xlabel("Epoch")
pyplot.ylabel("Loss")
pyplot.title("Loss over Epochs")
pyplot.legend()
pyplot.grid(True)

# --- Accuracy plot ---
pyplot.subplot(1, 2, 2)
pyplot.plot(train_acc_list, label="Train Accuracy", marker='o')
pyplot.plot(val_acc_list, label="Val Accuracy", marker='o')
pyplot.xlabel("Epoch")
pyplot.ylabel("Accuracy")
pyplot.title("Accuracy over Epochs")
pyplot.legend()
pyplot.grid(True)

pyplot.tight_layout()
pyplot.show()

## Step 8: Evaluation on Test Set

In this final step, we evaluate the performance of our trained model on the test set, which consists of completely unseen data — not used during training or validation.

While the validation set is typically used to experiment with hyperparameters (e.g., number of layers, hidden units, regularization methods, etc.), we now pass that stage to assess how well the final model generalizes on test set.

We perform the following:

- Run inference on the test data to get predicted class labels
- Calculate overall test accuracy to measure generalization performance
- Visualize a confusion matrix to analyze per-class prediction quality — helpful for identifying which characters the model confuses most often
- Visualize a few example predictions, including the predicted class and its probability, to better understand how the model behaves on real data. Please remember that the model outputs *raw logits*, not probabilities. To convert these logits into class probabilities, we need to apply the `Softmax` function using `torch.nn.functional.softmax`.

Please note that we make use of `sklearn` functionality directly using `torch.tensor` data structures — there is no need to convert them into `numpy` arrays. Just make sure that variables are stored on the CPU `device` and have no gradients attached.

In [ ]:
# Set model to evaluation mode
model.eval()

# Store predictions
test_predictions, test_probabilities, test_labels = [], [], []

# Disable gradient calculation for efficiency
with torch.no_grad():
    for x, t in test_loader:
        z = model(x.to(device))

        # Apply Softmax to get probabilities
        probabilities = torch.nn.functional.softmax(z, dim=1)

        # Get the index of highest probability for predicted class
        maximum_probabilities, predicted_classes = torch.max(probabilities,dim=1)

        # Since all processing was done at the GPU, we first need to move the resulting variables to the CPU
        test_predictions.extend(predicted_classes.cpu())
        test_probabilities.extend(maximum_probabilities.cpu())
        test_labels.extend(t)

# convert all lists to torch tensors
test_predictions = torch.tensor(test_predictions)
test_probabilities = torch.tensor(test_probabilities)
test_labels = torch.tensor(test_labels)

# --- Accuracy ---
test_accuracy = sklearn.metrics.accuracy_score(test_labels, test_predictions)
print(f"Test Accuracy: {test_accuracy:.4f}")

# --- Confusion Matrix ---
class_names = list(test_dataset.label2index.keys())
figure = pyplot.figure(figsize=(12, 12))
ax = figure.gca()
sklearn.metrics.ConfusionMatrixDisplay.from_predictions(
    test_labels,
    test_predictions,
    display_labels=class_names,
    ax=ax,
    xticks_rotation="vertical",
    text_kw={"fontsize":"x-small"}
)
ax.grid(False)


Let's see some correctly predicted test samples including their probabilites.

In [ ]:
torch.random.manual_seed(5)
# Map index to label (if your dataset has this mapping)
index2label = {v:k for k,v in test_dataset.label2index.items()}

# Pick random test samples to visualize
num_samples = 8
indices = torch.randperm(len(test_predictions))[:num_samples]

pyplot.figure(figsize=(14, 4))
for i, idx in enumerate(indices):
    sample_img = test_dataset.inputs[idx].reshape(32, 32)
    true_label = test_labels[idx].item()
    pred_label = test_predictions[idx].item()
    confidence = test_probabilities[idx] * 100

    pyplot.subplot(2, 4, i+1)
    pyplot.imshow(sample_img)
    pyplot.axis('off')
    pyplot.title(f"Pred: {index2label[pred_label]}\n"
                 f"True: {index2label[true_label]}\n"
                 f"Conf: {confidence:.1f}%")

pyplot.tight_layout()
pyplot.show()

Let's also show some test samples that were predicted incorrectly, together with their probability and a sample from the predicted and the correct class.

In [ ]:
torch.random.manual_seed(20)
# Select a few random misclassified indices
misclassified = torch.where(test_labels != test_predictions)[0]
# select four random samples
sampled = misclassified[torch.randperm(misclassified.shape[0])[:4]]

pyplot.figure(figsize=(15, 5))

for i, idx in enumerate(sampled):
    img = test_dataset.inputs[idx].reshape(32, 32)
    true_label = test_labels[idx].item()
    pred_label = test_predictions[idx].item()
    prob = test_probabilities[idx] * 100

    # Get one true image from the predicted class
    pred_match_idxs = torch.where(test_labels == pred_label)[0]
    pred_example_idx = torch.randint(pred_match_idxs.shape[0], (1,))
    pred_example_img = test_dataset.inputs[pred_example_idx].reshape(32, 32)

    # Get one true image from the true class
    true_match_idxs = torch.where(test_labels == true_label)[0]
    true_example_idx = torch.randint(true_match_idxs.shape[0], (1,))
    true_example_img = test_dataset.inputs[true_example_idx].reshape(32, 32)

    # --- Plot original misclassified image ---
    pyplot.subplot(3, len(sampled), i + 1)
    pyplot.imshow(img, cmap='gray')
    pyplot.axis('off')
    pyplot.title(f"Pred: {index2label[pred_label]}\nTrue: {index2label[true_label]}\nConf: {prob:.1f}%")

    # --- Plot one sample from true class ---
    pyplot.subplot(3, len(sampled), i + 1 + len(sampled))
    pyplot.imshow(true_example_img, cmap='gray')
    pyplot.axis('off')
    pyplot.title(f"Example of True: {index2label[true_label]}")

    # --- Plot one sample from predicted class ---
    pyplot.subplot(3, len(sampled), i + 1 + 2*len(sampled))
    pyplot.imshow(pred_example_img, cmap='gray')
    pyplot.axis('off')
    pyplot.title(f"Example of Pred: {index2label[pred_label]}")

pyplot.tight_layout()
pyplot.show()


## Summary

In this project, we developed a deep learning pipeline to classify handwritten Devanagari characters using `PyTorch`. 
The dataset consists of flattened grayscale images stored in a .csv file, representing *46 unique character classes*. 
Key steps in the pipeline were:

- **Dataset Preparation**: We preprocessed the data, normalized pixel values, encoded string labels into integers, and organized samples into batches using `torch.utils.data.Dataset` and `torch.utils.data.DataLoader` — demonstrating PyTorch’s flexibility and simplicity for custom data handling.
- **Model Definition**: We implemented a fully connected neural network using `torch.nn.Module`, showing how easily and flexibly PyTorch allows you to build and modify your own model architectures.
- **Loss & Optimizer Setup**: We used `torch.nn.CrossEntropyLoss` and `torch.optim.SGD`, which are straightforward to implement and configure for standard classification tasks.
- **Training Loop**: PyTorch provides a clean and intuitive structure for training — computing gradients with `loss.backward()` and updating weights with `optimizer.step()` is easy to follow. Just do never forget to call `optimizer.zero_grad()` after an update step.
- **Evaluation**: We combined data visualization with model predictions and provided valuable insights into where the model succeeds or struggles.

💡 **Takeaways**

PyTorch makes deep learning workflows modular, readable, and easy to customize at every stage.

Writing a training and evaluation loop with PyTorch is both powerful and beginner-friendly — gradients and weight updates are handled transparently.